In [ ]:
# Install the packages we need for this notebook.
# datasets = lets us load the Hugging Face dataset
# pyarrow = lets pandas read/write parquet files
!pip install datasets pyarrow scikit-learn pandas

In [ ]:
# import packages
import pandas as pd
import numpy as np
import json
import os

from datasets import load_dataset
from sklearn.model_selection import train_test_split

In [ ]:
# set seed and output folder
SEED = 42
OUTPUT_DIR = "data_splits"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# load dataset from hugging face
dataset = load_dataset("ai4privacy/pii-masking-400k", split="train")

# turn dataset into a pandas dataframe
df = dataset.to_pandas()

# check size and columns
print(len(df))
print(df.columns.tolist())

In [ ]:
# show all column names
df.columns.tolist()

In [ ]:
# look at first few rows
df.head()

**Dataset inspection notes**

The dataset has original text, masked text, privacy masks, and token-level PII labels. The main input text appears to be source_text. The token-level PII labels are in mbert_token_classes.

In [ ]:
# look at one row clearly
row = df.iloc[0]

print("source text:")
print(row['source_text'])

print("\nprivacy mask:")
print(row['privacy_mask'])

print("\nmbert token classes:")
print(row['mbert_token_classes'])

**Single-row inspection notes**

This example shows how the dataset stores PII. The `source_text` column contains the original text. The `privacy_mask` column lists the PII categories and values found in the text. The `mbert_token_classes` column gives token-level labels, where `O` means not PII and labels like `B-EMAIL` or `I-EMAIL` mark PII tokens.

For the main binary task, any row with a token label other than `O` should be labeled privacy-sensitive.

In [ ]:
# make binary safe/pii label
def is_sensitive(row):
    tags = row.get('mbert_token_classes', None)

    if tags is not None:
        return int(any(str(t).strip() != 'O' for t in tags))

    return 0

# apply label function to every row
df['label'] = df.apply(is_sensitive, axis=1)

In [ ]:
# check label counts
df['label'].value_counts()

In [ ]:
# check label percentages
df['label'].value_counts(normalize=True) * 100

**Binary label check**

Using `mbert_token_classes`, I created a binary label where any token label other than `O` becomes `1` = `privacy-sensitive`. Rows with only `O` labels become `0 = safe`.

The resulting class balance is about 67.4% sensitive and 32.6% safe, which matches the split metadata. This confirms that the notebook is using the same label-collapse logic as `src/data/data_split.py`.

In [ ]:
# keep only columns needed for main binary task
df_core = df[['source_text', 'label']].copy()

# rename source_text to text so it is easier to use later
df_core.columns = ['text', 'label']

# keep original row number so we can connect back to pii categories later
df_core['original_index'] = df.index

# drop rows where text is missing
df_core = df_core.dropna(subset=['text'])

# check the cleaned dataframe
df_core.head()

**Clean modeling dataframe**

I created a smaller dataframe for the main binary classification task. It keeps only the prompt text, the binary label, and the original row index. The original_index column is important because it lets us reconnect back to the full dataset later for PII category analysis.

In [ ]:
# make train and temp split
train_df, temp_df = train_test_split(
    df_core,
    test_size=0.20,
    random_state=SEED,
    stratify=df_core['label']
)

# split temp into validation and test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df['label']
)

In [ ]:
# check split sizes
print("train:", train_df.shape)
print("validation:", val_df.shape)
print("test:", test_df.shape)

### **Frozen split check**

I recreated the frozen train/validation/test split using the same settings as src/data/data_split.py: 80/10/10 split, stratified by the binary label, with seed 42.

The split sizes are:

- train: 260,413 rows
- validation: 32,552 rows
- test: 32,552 rows

This confirms the notebook can reproduce the same split locally.

In [ ]:
# check class balance for each split
for name, split_df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    print(name)
    print(split_df['label'].value_counts(normalize=True) * 100)
    print()

### **Split class balance check**

The train, validation, and test splits all keep nearly the same class balance: about 67.4% sensitive and 32.6% safe. This confirms that the stratified split worked correctly.

This matters because all models should be trained and evaluated on splits with similar label distributions.

In [ ]:
# check overlap between splits
train_texts = set(train_df['text'])
val_texts = set(val_df['text'])
test_texts = set(test_df['text'])

print("train and validation overlap:", len(train_texts.intersection(val_texts)))
print("train and test overlap:", len(train_texts.intersection(test_texts)))
print("validation and test overlap:", len(val_texts.intersection(test_texts)))

In [ ]:
# check duplicate text in full dataset
duplicate_count = df_core['text'].duplicated().sum()

print("duplicate text rows:", duplicate_count)
print("unique text rows:", df_core['text'].nunique())
print("total rows:", len(df_core))

In [ ]:
# look at duplicate rows without printing full text
duplicate_rows = df_core[df_core['text'].duplicated(keep=False)]

duplicate_rows[['label', 'original_index']].head(10)

### **Duplicate text check**

The full dataset contains 93 duplicate text rows out of 325,517 total rows. Because splitting was done at the row level, a small number of duplicate texts appear across train, validation, and test splits.

This is a possible data leakage issue, but the overlap is very small compared to the full dataset. I am documenting it so the group can decide whether to keep the frozen split as-is or deduplicate before splitting.

In [ ]:
# add simple text length columns
df_core['char_count'] = df_core['text'].str.len()
df_core['word_count'] = df_core['text'].str.split().str.len()

In [ ]:
# compare text length by label
df_core.groupby('label')[['char_count', 'word_count']].describe()

**Main observation**

Privacy-sensitive examples are slightly longer on average than safe examples.

- Safe examples average about 133 characters and 17.5 words.
- Sensitive examples average about 157 characters and 19.8 words.

This means text length may be somewhat related to the label, but the difference is not large enough to rely on by itself.

In [ ]:
# Check rows where the text is empty or missing
empty_text_rows = df_core[df_core['char_count'] == 0]

# Show how many empty-text rows we found
print("Number of empty text rows:", len(empty_text_rows))

# Display a few empty-text rows so we can inspect them
empty_text_rows.head()

**Empty text rows**

I found 6 rows where the text field is empty. All of these rows are labeled safe (`label = 0`). Since this is only a tiny number of examples compared with the full dataset, it should not strongly affect the dataset overall. However, empty text rows may be removed before modeling because they do not contain useful language features.

In [ ]:
# Sort the dataset by longest text first
longest_rows = df_core.sort_values(by='char_count', ascending=False)

# Look at the longest examples
longest_rows[['text', 'label', 'char_count', 'word_count']].head()

**Longest text rows**

The longest examples are mostly labeled privacy-sensitive and appear to contain HTML/form-like text. This makes sense because forms often include fields such as names, child names, guardian names, or other personal information. This also means the model may learn patterns from structured/form text, not only normal conversational text.

A limitation is that some privacy-sensitive examples are structured forms or HTML-like text. Models may partly learn formatting patterns associated with forms, rather than only detecting PII content itself.

In [ ]:
# Count how many examples are in each label
label_counts = df_core['label'].value_counts().sort_index()

# Convert label counts into percentages
label_percentages = df_core['label'].value_counts(normalize=True).sort_index() * 100

# Combine counts and percentages into one table
label_summary = pd.DataFrame({
    'count': label_counts,
    'percentage': label_percentages
})

# Display the label summary table
label_summary

**Label distribution**

The dataset is imbalanced. There are 106,257 safe examples (`label = 0`), which is about 32.6% of the dataset. There are 219,260 privacy-sensitive examples (`label = 1`), which is about 67.4% of the dataset.

This matters because accuracy alone could be misleading. A model could perform well overall by favoring the majority class, so later evaluation should include precision, recall, and F1 score, especially for the privacy-sensitive class.

In [ ]:
# Check how many missing values are in each column
missing_values = df_core.isnull().sum()

# Display the missing value counts
missing_values

### Missing values

There are no missing/null values in the current working columns (`text`, `label`, `original_index`, `char_count`, and `word_count`). This means the dataset does not have `NaN` values in these fields.

However, this is different from empty strings. Earlier, I found 6 rows where `text` is an empty string, even though those rows are not technically missing/null values.

In [ ]:
# Count how many duplicated text values exist in the dataset
num_duplicate_texts = df_core['text'].duplicated().sum()

# Print the number of duplicate text rows
print("Number of duplicate text rows:", num_duplicate_texts)

# Look at a few duplicated text rows
duplicate_text_rows = df_core[df_core['text'].duplicated(keep=False)].sort_values(by='text')

# Display a few duplicate rows
duplicate_text_rows[['text', 'label', 'original_index', 'char_count', 'word_count']].head(10)

### Duplicate text rows

I found 93 rows where the `text` value is an exact duplicate of an earlier row. Some duplicated text values are empty strings, while others are short structured snippets such as HTML fragments or repeated document phrases.

This is a small number compared with the full dataset, but duplicates are still important because they can create data leakage if the same exact text appears in both the training set and the validation/test set. Before modeling, we should confirm whether the frozen split keeps duplicate texts within the same split or whether duplicates appear across different splits.

In [ ]:
# Check the column names currently available in df_core
# We need to see whether this dataframe already knows which rows are train, validation, or test

df_core.columns

In [ ]:
# check if any duplicated text has more than one label

duplicate_label_check = (
    df_core[df_core["text"].duplicated(keep=False)]
    .groupby("text")
    .agg(
        num_rows=("text", "size"),
        num_labels=("label", "nunique"),
        labels=("label", lambda x: sorted(x.unique()))
    )
    .sort_values(by="num_labels", ascending=False)
)

duplicate_label_check.head(20)

In [ ]:
# show duplicate texts where labels disagree

conflicting_duplicate_labels = duplicate_label_check[duplicate_label_check["num_labels"] > 1]

print("Number of duplicate text groups with conflicting labels:", len(conflicting_duplicate_labels))

conflicting_duplicate_labels

### Duplicate label consistency

No duplicated text appeared under both classes.

This makes cleanup simpler because deduplication would only remove repeat copies, not force us to choose between conflicting safe vs privacy-sensitive labels.

In [ ]:
# preview how many rows basic cleaning would remove

blank_text_count = (df_core["text"].fillna("").str.strip() == "").sum()
duplicate_text_count = df_core["text"].duplicated().sum()

df_clean_preview = df_core[df_core["text"].fillna("").str.strip() != ""]
df_clean_preview = df_clean_preview.drop_duplicates(subset=["text"], keep="first")

print("blank text rows:", blank_text_count)
print("duplicate text rows after the first copy:", duplicate_text_count)
print("current total rows:", len(df_core))
print("rows after removing blanks and exact duplicates:", len(df_clean_preview))
print("rows removed:", len(df_core) - len(df_clean_preview))

### Basic cleaning preview

Removing blank text rows and exact duplicate text rows would remove 94 rows total. The dataset would go from 325,517 rows to 325,423 rows.

This is a very small change to the dataset, but it would make the data cleaner before splitting and reduce the risk of duplicate text leaking across train, validation, and test.

In [ ]:
# compare label balance before and after the cleaning preview

before_cleaning = df_core["label"].value_counts(normalize=True).sort_index() * 100
after_cleaning = df_clean_preview["label"].value_counts(normalize=True).sort_index() * 100

cleaning_label_comparison = pd.DataFrame({
    "before_cleaning_percent": before_cleaning,
    "after_cleaning_percent": after_cleaning
})

cleaning_label_comparison

### Cleaning and label balance

The basic cleaning preview does not meaningfully change the label distribution. Before cleaning, the dataset was about 32.6% safe and 67.4% privacy-sensitive. After removing blank text rows and exact duplicates, the percentages stay almost the same.

This means the cleaning step would make the dataset cleaner without noticeably changing the class balance.

In [ ]:
# compare text length by label after the cleaning preview

df_clean_preview.groupby("label")[["char_count", "word_count"]].describe()

### Text length after cleaning preview

After removing blank text rows and exact duplicates, privacy-sensitive examples are still slightly longer on average than safe examples.

This means the basic cleaning step does not really change the earlier text-length pattern. Length may be useful as a small feature, but it should not be treated as enough to detect PII by itself.

In [ ]:
# look at a few examples from each label

safe_examples = df_clean_preview[df_clean_preview["label"] == 0].sample(5, random_state=42)
sensitive_examples = df_clean_preview[df_clean_preview["label"] == 1].sample(5, random_state=42)

display(safe_examples[["text", "label", "char_count", "word_count"]])
display(sensitive_examples[["text", "label", "char_count", "word_count"]])

### Sample examples by label

I looked at a few random examples from each label to sanity check what the classes look like.

The safe examples mostly look like normal text snippets, although some still include URLs or formatting. The privacy-sensitive examples appear more likely to include personal context, forms, HTML-like structure, or placeholders for personal information.

This supports the main project setup, but it also shows a limitation: the model may learn formatting or placeholder patterns in addition to learning what privacy-sensitive text looks like.

In [ ]:
# check how often each row has pii style placeholder tokens

df_clean_preview["has_placeholder"] = df_clean_preview["text"].str.contains(r"\[[A-Z]+(?:_[A-Z]+)*_\d+\]", regex=True)

placeholder_summary = (
    df_clean_preview
    .groupby("label")["has_placeholder"]
    .agg(["count", "sum", "mean"])
)

placeholder_summary["percentage"] = placeholder_summary["mean"] * 100

placeholder_summary

### Placeholder token check

I checked how often examples contain bracket-style placeholder tokens, such as names, emails, phone numbers, or similar structured placeholders.

The placeholder pattern appears in both classes and is actually more common in the safe class than the privacy-sensitive class in this check. This means the label is not simply based on whether the text contains bracketed placeholder tokens.

This is useful because it reduces one concern about the dataset, but it does not remove all formatting concerns. Some examples still include forms, HTML, or structured text that models may learn from.

In [ ]:
# check how often text has html-like formatting

df_clean_preview["has_html_like_text"] = df_clean_preview["text"].str.contains(r"<[^>]+>", regex=True)

html_summary = (
    df_clean_preview
    .groupby("label")["has_html_like_text"]
    .agg(["count", "sum", "mean"])
)

html_summary["percentage"] = html_summary["mean"] * 100

html_summary

### HTML-like text check

HTML-like formatting appears in both classes, but it is more common in the privacy-sensitive examples.

This supports the earlier observation from the longest rows. Some privacy-sensitive examples look like forms or structured text, so the model may learn formatting patterns along with actual PII patterns.

### EDA findings and recommendations

This notebook reviewed the AI4Privacy dataset before modeling. I checked the binary label setup, split quality, missing and blank text, duplicate rows, text length, sample examples, placeholder tokens, and HTML-like formatting.

#### What I checked and found

**Binary label setup:**  
I created a binary label from `mbert_token_classes`. Rows with any non-`O` token label are privacy-sensitive (`label = 1`), and rows with only `O` labels are safe (`label = 0`). This produced 219,260 privacy-sensitive examples and 106,257 safe examples.

**Label distribution:**  
The dataset is imbalanced. About 67.4% of examples are privacy-sensitive and about 32.6% are safe. This means accuracy alone will not be enough for evaluation. Later model results should include precision, recall, and F1 score.

**Split quality:**  
The 80/10/10 stratified split produced 260,413 training rows, 32,552 validation rows, and 32,552 test rows. The label balance stayed almost identical across all three splits, which shows the stratified split worked correctly.

**Duplicate leakage risk:**  
There are 93 duplicate text rows after the first copy. Because the split was row-based, a small number of exact duplicate texts appear across split boundaries: 11 train-validation overlaps, 9 train-test overlaps, and 3 validation-test overlaps. This is small compared with the full dataset, but it is still a possible leakage issue.

**Missing and blank text:**  
There are no missing/null values in the core columns. However, there are 6 blank text rows. These rows are all labeled safe and do not contain useful language features.

**Duplicate label consistency:**  
No duplicated text appeared under both classes. This means the duplicate rows are label-consistent, so deduplication would not require choosing between conflicting safe vs privacy-sensitive labels.

**Cleaning impact:**  
Removing blank text rows and exact duplicate text rows would remove 94 rows total. The dataset would go from 325,517 rows to 325,423 rows. This is a very small change and does not meaningfully change the label balance or text-length pattern.

**Text length:**  
Privacy-sensitive examples are slightly longer on average than safe examples. Safe examples average about 133 characters and 17.5 words, while privacy-sensitive examples average about 157 characters and 19.8 words. Text length may be a small useful feature, but it is not enough by itself to detect PII.

**Sample examples:**  
The random samples show that safe examples mostly look like normal text snippets, although some include URLs or formatting. Privacy-sensitive examples are more likely to include personal context, forms, HTML-like structure, or PII-related content.

**Placeholder tokens:**  
Bracket-style placeholder tokens appear in both classes and are actually more common in the safe class in this check. This means the binary label is not simply based on whether text contains bracketed placeholder tokens.

**HTML-like formatting:**  
HTML-like formatting appears in both classes, but it is more common in privacy-sensitive examples. About 16.9% of safe examples and 26.0% of privacy-sensitive examples contain HTML-like formatting. This is useful signal, but it is also a limitation because models may learn formatting patterns along with actual PII patterns.

#### Final recommendations

Before final modeling, I recommend applying a small cleaning step:

1. Remove the 6 blank text rows.
   - These rows are labeled safe and do not contain useful language features.

2. Remove exact duplicate text rows.
   - There are 93 duplicate text rows after the first copy.
   - I checked for duplicate label conflicts and found 0 conflicting duplicate-label groups.

3. Regenerate the train/validation/test split after cleaning.
   - The previous split was row-based, so a small number of duplicate texts crossed split boundaries.

4. Keep the regenerated split stratified by the binary label.
   - The current stratified split keeps label balance nearly identical across train, validation, and test.

5. Recheck the new split quality.
   - Check row counts, label balance, duplicate overlap, and the updated `split_metadata.json`.

This would remove only 94 rows out of 325,517, so it should not materially change the dataset. The main benefit is reducing the duplicate text leakage risk before modeling.